In [1]:
import pathlib
from argparse import ArgumentParser
import yaml
import torch
import torchmetrics
import pytorch_lightning as pl
from tqdm import tqdm

/om4/group/mcdermott/user/imgriff/conda_envs_files/torch_11_cuda_11/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys 
sys.path.append('../')
from src.attn_tracking_lightning import AttentionalTrackingModule
from corpus.jsinV3AttnTrackingValidation import jsinV3_attn_tracking_validation
import src.audio_transforms as at


In [3]:
pl.seed_everything(1) 

Global seed set to 1


1

In [4]:
path = '../config/attentional_cue/attn_cue_lr_1e-4_bs_64_constrain_slope.yml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)

In [5]:
config['data']['audio']['rep_kwargs']['rep_on_gpu'] = False

In [6]:

audio_config = config['data']['audio']

audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.CombineWithRandomDBSNR(low_snr=0, high_snr=0), # set to 0 so foreground/background at same level 
            at.RMSNormalizeForegroundAndBackground(rms_level=0.1)
])

cochgram_transforms = at.AudioCompose([
            at.UnsqueezeAudio(dim=0),
            at.AudioToAudioRepresentation(**audio_config),
            at.UnsqueezeAudio(dim=0),

])

In [7]:
model = AttentionalTrackingModule(config)
dataset = jsinV3_attn_tracking_validation(**config['data']['corpus'],
                                          train=False,
                                          transform=audio_transforms,
                                          demo=True)

In [8]:
ckpt_path = '../attn_cue_models/attn_cue_jsin_pilot_no_pretrain_pos_slope_bs_64_lr_1e-4/checkpoints/epoch=1-step=145791.ckpt'
ckpt = torch.load(ckpt_path, map_location=torch.device('cpu'))

model.load_state_dict(ckpt['state_dict'])

<All keys matched successfully>

In [9]:
# fg_acc = torchmetrics.Accuracy()
# bg_acc = torchmetrics.Accuracy()

In [9]:
model = model.eval()
# fg_acc.cuda()
# bg_acc.cuda()


In [11]:
model.device.type

'cpu'

In [12]:
# val_loader = torch.utils.data.DataLoader(dataset, batch_size=1, num_workers=0)

In [13]:
def get_model_pred(transforms, model, word_label_map, 
                   mixture, foreground_cue, background_cue):
    # process wavs to get cochleagrams
    mixture_coch, _ = transforms(mixture, None)
    fg_cue_coch, _ = transforms(foreground_cue, None)
    bg_cue_coch, _ = transforms(background_cue, None)
    
    # get model device
    device = model.device
    
    # pass sounds through model
    fg_out = model(fg_cue_coch.to(device), mixture_coch.to(device))
    bg_out = model(bg_cue_coch.to(device), mixture_coch.to(device))
    
    # get model predictions
    model_fg_pred = fg_out.log_softmax(-1).argmax(-1)
    model_fg_pred = word_map[model_fg_pred.item()]
    
    model_bg_pred = bg_out.log_softmax(-1).argmax(-1)
    model_bg_pred = word_map[model_bg_pred.item()]
    
    return model_fg_pred, model_bg_pred
    
    
    

In [14]:
from IPython.display import Audio

In [15]:
word_map = dataset.class_map()

### Demo: 0dB SNR 

In [30]:
foreground, background, mixture, fg_cue, bg_cue, fg_target, bg_target = dataset[20]

In [31]:
Audio(fg_cue, rate=20000)

In [32]:
Audio(mixture, rate=20000)

In [33]:
Audio(bg_cue, rate=20000)

In [34]:
Audio(foreground, rate=20000)

In [35]:
Audio(background, rate=20000)

In [36]:
print(f"Foreground word: {word_map[fg_target]} \nBackground word: {word_map[bg_target]}")
model_fg_pred, model_bg_pred = get_model_pred(cochgram_transforms, model, word_map,
                                             mixture, fg_cue, bg_cue)

print(f"\nForeground prediction: {model_fg_pred}\nBackground prediction: {model_bg_pred}")

Foreground word: damage 
Background word: forced

Foreground prediction: damage
Background prediction: eventually


### Demo at high snr

In [22]:
audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.CombineWithRandomDBSNR(low_snr=5, high_snr=10), # set to 0 so foreground/background at same level 
            at.RMSNormalizeForegroundAndBackground(rms_level=0.1)
])
dataset = jsinV3_attn_tracking_validation(**config['data']['corpus'],
                                          train=False,
                                          transform=audio_transforms,
                                          demo=True)

In [23]:
foreground, background, mixture, fg_cue, bg_cue, fg_target, bg_target = dataset[100]

In [24]:
Audio(fg_cue, rate=20000)

In [25]:
Audio(mixture, rate=20000)

In [26]:
Audio(bg_cue, rate=20000)

In [27]:
Audio(foreground, rate=20000)

In [28]:
Audio(background, rate=20000)

In [29]:
print(f"Foreground word: {word_map[fg_target]} \nBackground word: {word_map[bg_target]}")
model_fg_pred, model_bg_pred = get_model_pred(cochgram_transforms, model, word_map,
                                             mixture, fg_cue, bg_cue)

print(f"\nForeground prediction: {model_fg_pred}\nBackground prediction: {model_bg_pred}")

Foreground word: powers 
Background word: corporation

Foreground prediction: powers
Background prediction: corporation


### Demo at low SNR

In [45]:
audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
            at.CombineWithRandomDBSNR(low_snr=-15, high_snr=-15), # set to 0 so foreground/background at same level 
])
dataset = jsinV3_attn_tracking_validation(**config['data']['corpus'],
                                          train=False,
                                          transform=audio_transforms,
                                          demo=True)

In [58]:
foreground, background, mixture, fg_cue, bg_cue, fg_target, bg_target = dataset[30]

In [59]:
Audio(fg_cue, rate=20000)

In [60]:
Audio(mixture, rate=20000)

In [61]:
Audio(bg_cue, rate=20000)

In [62]:
Audio(foreground, rate=20000)

In [63]:
Audio(background, rate=20000)

In [64]:
print(f"Foreground word: {word_map[fg_target]} \nBackground word: {word_map[bg_target]}")
model_fg_pred, model_bg_pred = get_model_pred(cochgram_transforms, model, word_map,
                                             mixture, fg_cue, bg_cue)

print(f"\nForeground prediction: {model_fg_pred}\nBackground prediction: {model_bg_pred}")

Foreground word: power 
Background word: during

Foreground prediction: however
Background prediction: years
